[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_5/13_LLMs.ipynb)


# LLMs — capacidades cognitivas con Gemini

En el **12** recuperaste páginas con embeddings + FAISS. Aquí el foco es el **modelo de lenguaje** (LLM): cómo hablarle de forma estructurada y qué más puede hacer (ver imágenes, usar herramientas, devolver JSON).

**Idea central:** un LLM no recibe solo “un texto suelto”. Recibe una **conversación** con **roles** — el **prompt de sistema**, los mensajes del **usuario** y las respuestas previas del **modelo**.

Usamos la **Gemini API** (`google-genai`) y la misma clave que en los notebooks **11** y **12**.

| Sección | Tema |
|---------|------|
| **1** | Clave API (local vs Colab) |
| **2** | Roles y prompts — lo más importante |
| **3** | Tokens |
| **4** | Visión (imagen + texto) |
| **5** | Function calling |
| **6** | Búsqueda en Google |
| **7** | JSON estructurado (expediente médico) |


## Setup — instalar librerías

- `google-genai`, `python-dotenv`, `requests`, `pypdf` (para leer el expediente en la sección 7)


In [ ]:
# Comentar esta línea si estás en local
# %pip install -q google-genai python-dotenv requests pypdf


In [ ]:
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pypdf import PdfReader

BASE_DIR = Path(".")


## 1 — Clave API (igual que notebooks 11 y 12)

**Local (Cursor / Jupyter):** descomenta y ejecuta la celda **Local**. Necesitas un `.env` en esta carpeta con `GOOGLE_API_KEY=...` ([AI Studio](https://aistudio.google.com/apikey)).

**Google Colab:** ejecuta solo la celda **Colab** (la de abajo). Guarda la clave en *Secrets* con el nombre `GOOGLE_API_KEY`.

> No ejecutes las dos celdas a la vez — usa la que corresponda a tu entorno.


In [ ]:
# --- Local: descomenta todo este bloque ---
load_dotenv(BASE_DIR / ".env", override=True)

API_KEY = (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY") or "").strip()
if not API_KEY:
    raise ValueError("Falta GOOGLE_API_KEY en .env — https://aistudio.google.com/apikey")

cliente = genai.Client(api_key=API_KEY)
MODELO = "gemini-2.5-flash"
print(f"Cliente listo. Modelo: {MODELO}")


In [ ]:
# # --- Colab: ejecuta esta celda (Secrets → GOOGLE_API_KEY) ---
# from google.colab import userdata
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# cliente = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
# MODELO = "gemini-2.5-flash"

# print(f"Cliente listo. Modelo: {MODELO}")


In [ ]:
VENTANA = 10  # pares user+model que se mandan (el system va aparte, en cada llamada)

def contexto(conversacion):
    return conversacion[-VENTANA * 2:]


def chat(conversacion, entrada, system=None):
    """Lista + mensaje nuevo → respuesta. Guarda todo; a la API solo va la ventana."""
    conversacion.append({"role": "user", "content": entrada})
    contents = [
        types.Content(role=m["role"], parts=[types.Part.from_text(text=m["content"])])
        for m in contexto(conversacion)
    ]
    if system:
        resp = cliente.models.generate_content(
            model=MODELO,
            contents=contents,
            config=types.GenerateContentConfig(system_instruction=system),
        )
    else:
        resp = cliente.models.generate_content(model=MODELO, contents=contents)
    conversacion.append({"role": "model", "content": resp.text})
    return resp


## 2 — ¿Qué es un “prompt” en un LLM?

Cuando escribes en un chatbot, en realidad estás armando **mensajes con rol**. La API de Gemini funciona igual.

| Rol | Nombre en Gemini | ¿Quién lo escribe? | Para qué sirve |
|-----|------------------|--------------------|----------------|
| **Sistema** | `system_instruction` (va aparte del historial) | Tú (desarrollador) | Reglas fijas: tono, personalidad, límites. El usuario **no** lo ve en el chat, pero el modelo **sí** lo lee en cada llamada. |
| **Usuario** | `role="user"` | La persona que pregunta | La duda, la imagen o la orden del momento. |
| **Modelo** | `role="model"` | El LLM (turnos anteriores) | Respuestas que ya dio el asistente; sirven para **continuar** la conversación. |

**Dos formas de mandar texto a Gemini**

| Forma | Cuándo usarla | Ejemplo |
|-------|---------------|---------|
| **Corta** — un string en `contents` | Una pregunta de solo texto | `contents="Hola"` |
| **Explícita** — lista de dicts `{role, content}` | Historial multi-turno (sección 2.3) | `[{"role": "user", "content": "Hola"}]` |
| **Explícita** — `types.Content` + `parts` | Texto **+** imagen (sección 4) | Ver sección 4 |

> El atajo `contents="Hola"` hace lo mismo por dentro que armar un mensaje `user` con `parts`; la forma explícita la verás cuando haga falta mezclar tipos de contenido (historial o imagen).


### 2.1 — Un mensaje de usuario (forma corta)

Lo más simple: un string en `contents`. Gemini lo trata como un mensaje `user` sin system ni historial.


In [ ]:
# Forma corta: basta un string para texto simple
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="Hola! ¿Cómo estás?",
)
print(respuesta.text)


### 2.2 — Prompt de **sistema** (`system_instruction`)

El **system** es el manual del asistente: identidad, tono y límites. Va en `config`, **no** en el historial del chat.

Un buen prompt de sistema suele incluir:

- **Objetivo** — para qué existe el bot
- **Tareas** — qué puede hacer (y qué no)
- **Tono** — cómo debe hablar
- **Límites** — qué debe evitar (inventar datos, dar consejo médico, etc.)

Aquí la pregunta del usuario sigue siendo un string; solo añadimos el system.


In [ ]:
SYSTEM_IBERIA = """
Eres Iber.ia, asistente virtual de la Universidad Iberoamericana Ciudad de México.

OBJETIVO
Ayudar a alumnos con dudas académicas y administrativas de forma clara y confiable.

TAREAS
- Responder preguntas sobre inscripción, horarios, biblioteca, servicios escolares y trámites comunes.
- Orientar al alumno: si falta un dato, pide contexto (carrera, semestre, campus).
- Si no sabes algo con certeza, dilo y sugiere contactar la mesa de ayuda o el departamento correspondiente.

TONO
Español, cercano pero profesional. Respuestas breves (3–6 oraciones salvo que pidan detalle).

LÍMITES
- No inventes fechas, costos, reglamentos ni disponibilidad de materias.
- No des consejo médico, legal ni psicológico.
- No compartas datos personales de otras personas.
"""

pregunta = "¿Quién eres y en qué me puedes ayudar?"

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=pregunta,  # string simple — igual que en 2.1
    config=types.GenerateContentConfig(system_instruction=SYSTEM_IBERIA),
)
print(respuesta.text)


**Conclusión:** el mensaje del usuario puede ser un string simple; el **system** define identidad, alcance y reglas. En la sección 2.3 usamos `SYSTEM_IBERIA` con una conversación en dicts.


### 2.3 — Chat interactivo (multi-turno)

Escribe en el `input`. En **cada llamada** se imprime lo que recibe Gemini: `system` + ventana del historial + tu mensaje. Escribe `salir` para terminar.

La lista `conversacion` guarda todo; a la API solo van los últimos `VENTANA` pares (igual que `01_ollama_chat.py`).


In [ ]:
conversacion = []

while True:
    entrada = input("Tú: ")
    if entrada.lower() == "salir":
        break

    # --- lo que va en ESTA llamada ---
    print("\n" + "─" * 50)
    print("Lo que va a Gemini:")
    print(f"  [system] {SYSTEM_IBERIA.strip()}")
    for m in contexto(conversacion) + [{"role": "user", "content": entrada}]:
        print(f"  [{m['role']}] {m['content']}")
    print("─" * 50)

    respuesta = chat(conversacion, entrada, system=SYSTEM_IBERIA)
    print(f"\nIber.ia: {respuesta.text}\n")


## 3 — Tokens

Cada llamada cobra por **entrada** (system + ventana del historial + pregunta) y **salida** (respuesta). Usa `respuesta` del último turno del chat de arriba.

A más historial, más tokens de entrada — por eso en producción se limita la ventana de contexto.


In [ ]:
u = respuesta.usage_metadata
print(f"Entrada (system + historial + pregunta): {u.prompt_token_count}")
print(f"Salida (respuesta del modelo):          {u.candidates_token_count}")
print(f"Total:                                {u.total_token_count}")


## 4 — Visión: imagen + texto

Cuando el mensaje lleva **imagen y texto**, no basta un string: hace falta `parts` con una pieza por tipo.

Usamos una copia del Partenón en este repositorio (`parthenon_athens.jpg`). En **Colab** se descarga desde GitHub; en **local** se lee del disco.

> **Nota:** Si descargas directo de Wikimedia en un aula con muchos alumnos en Colab, a veces aparece **429 Too Many Requests** — es límite de tráfico de Wikimedia, **no** un problema de tu `GOOGLE_API_KEY` ni de Gemini.


In [ ]:
RUTA_IMAGEN = BASE_DIR / "parthenon_athens.jpg"
URL_IMAGEN = (
    "https://raw.githubusercontent.com/milioe/casos-ia-ibero-diplomado/main/"
    "modulo_5/parthenon_athens.jpg"
)

if RUTA_IMAGEN.exists():
    bytes_img = RUTA_IMAGEN.read_bytes()
    print(f"Imagen local: {len(bytes_img) / 1024:.0f} KB")
else:
    resp = requests.get(URL_IMAGEN, timeout=60)
    resp.raise_for_status()
    bytes_img = resp.content
    print(f"Imagen descargada desde GitHub: {len(bytes_img) / 1024:.0f} KB")

if not bytes_img.startswith(b"\xff\xd8\xff"):
    raise ValueError(f"No llegó un JPEG válido ({len(bytes_img)} bytes)")

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_bytes(data=bytes_img, mime_type="image/jpeg"),  # pieza 1: imagen
                types.Part.from_text(text="Describe en español qué ves (2–4 oraciones)."),  # pieza 2: texto
            ],
        )
    ],
)
print(respuesta.text)


## 5 — Function calling

El **usuario** pide el clima; el **modelo** decide invocar tu función `get_weather`. La librería puede encadenar la llamada sola (`automatic_function_calling`).


In [ ]:
def get_weather(latitude: float, longitude: float) -> dict:
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    )
    temp = requests.get(url, timeout=30).json()["current"]["temperature_2m"]
    return {"latitude": latitude, "longitude": longitude, "temperature_c": temp}

print(get_weather(19.29, -99.18))


In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="¿Temperatura aproximada en Tlalpan, CDMX? Usa coordenadas razonables.",
    config=types.GenerateContentConfig(
        tools=[get_weather],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=3),
    ),
)
print(respuesta.text)


## 6 — Búsqueda en Google (grounding)

El modelo puede **buscar en la web** y citar fuentes. Documentación: [Google Search grounding](https://ai.google.dev/gemini-api/docs/google-search).


In [ ]:
respuesta = cliente.models.generate_content(
    model=MODELO,
    contents="Menciona 3 hechos recientes sobre adopción de IA en empresas mexicanas. Cita fuentes si puedes.",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())],
    ),
)
print(respuesta.text)


## 7 — Respuesta estructurada (JSON): expediente médico

En lugar de parsear a mano un PDF, le pasamos el **texto** del documento al LLM y pedimos salida **JSON** con campos fijos (`response_schema`).

Archivo de ejemplo en esta carpeta: `expediente_Rivera_Jose.pdf` (Hospital Estrella del Norte).


### 7.1 — Leer el PDF a texto


In [ ]:
RUTA_EXPEDIENTE = BASE_DIR / "expediente_Rivera_Jose.pdf"
if not RUTA_EXPEDIENTE.exists():
    raise FileNotFoundError(f"No encontré {RUTA_EXPEDIENTE.name}")

reader = PdfReader(RUTA_EXPEDIENTE)
texto_expediente = "\n\n".join(
    (page.extract_text() or "").strip() for page in reader.pages
)

print(f"Páginas: {len(reader.pages)}  |  Caracteres: {len(texto_expediente)}")
print(texto_expediente[:600], "...\n")


### 7.2 — Extraer campos con esquema JSON

Definimos qué campos queremos. En **Diagnóstico** del expediente hay varias enfermedades en un párrafo; pedimos **una fila por diagnóstico** en un arreglo `diagnosticos` (como filas de una tabla).


In [ ]:
esquema_expediente = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "nombre_completo": types.Schema(type=types.Type.STRING),
        "curp": types.Schema(type=types.Type.STRING),
        "fecha_nacimiento": types.Schema(type=types.Type.STRING),
        "edad": types.Schema(type=types.Type.INTEGER),
        "sexo": types.Schema(type=types.Type.STRING),
        "tipo_sangre": types.Schema(type=types.Type.STRING),
        "telefono": types.Schema(type=types.Type.STRING),
        "alergias": types.Schema(type=types.Type.STRING),
        "enfermedades_cronicas": types.Schema(type=types.Type.STRING),
        "medicamentos_actuales": types.Schema(type=types.Type.STRING),
        "ultima_visita": types.Schema(
            type=types.Type.OBJECT,
            description="Visita con fecha más reciente del expediente",
            properties={
                "fecha": types.Schema(type=types.Type.STRING),
                "motivo": types.Schema(type=types.Type.STRING),
                "diagnosticos": types.Schema(
                    type=types.Type.ARRAY,
                    description="Un elemento por cada diagnóstico (una fila c/u)",
                    items=types.Schema(type=types.Type.STRING),
                ),
            },
            required=["fecha", "diagnosticos"],
        ),
    },
    required=[
        "nombre_completo",
        "curp",
        "alergias",
        "enfermedades_cronicas",
        "medicamentos_actuales",
        "ultima_visita",
    ],
)

prompt_extraccion = f"""
Extrae la información del siguiente expediente médico.
Usa solo lo que aparece en el texto; si un campo no está, usa null o cadena vacía.

Para ultima_visita usa la visita con fecha más reciente.
En diagnosticos: separa cada diagnóstico en un elemento distinto del arreglo
(no los juntes en un solo párrafo). Ejemplo de formato (valores inventados):
- Rinofaringitis aguda
- Lumbalgia mecánica inespecífica
- Deficiencia de vitamina D

--- EXPEDIENTE ---
{texto_expediente}
--- FIN ---
"""

respuesta = cliente.models.generate_content(
    model=MODELO,
    contents=prompt_extraccion,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=esquema_expediente,
    ),
)

datos_paciente = json.loads(respuesta.text)
print(json.dumps(datos_paciente, indent=2, ensure_ascii=False))

print("\n--- Diagnósticos (una fila c/u) ---")
for i, dx in enumerate(datos_paciente["ultima_visita"]["diagnosticos"], start=1):
    print(f"{i}. {dx}")


## 8 — Cierre

| Ya viste | Siguiente paso |
|----------|----------------|
| **System + user + model** | Diseñar el system de tu chatbot (tono, límites, citas) |
| **Historial** | Guardar sesión en lista o base de datos |
| **Visión / tools / JSON** | Tickets, clima, **expedientes médicos** en JSON |
| **Notebook 12 (RAG)** | Recuperar páginas con FAISS y pasar el texto al LLM como mensaje **user** para la respuesta final con evidencia |

Eso es un asistente completo: **recuperar** (12) + **razonar y redactar** (este cuaderno).
